# Week 4 Task 3 — Reproducible Fare Prediction Pipeline

## Goal

Build a complete Spark ML workflow for trip-level fare prediction using the training Delta table from Task 1 and the feature-engineering design from Task 2.

This notebook:

- loads and validates the chronological Train, Validation, and Test splits;
- fits all learned feature transformations on Train only;
- trains a regularized Spark `LinearRegression` model;
- evaluates both the model and a Train-mean baseline using MAE, RMSE, and R²;
- saves one complete fitted `PipelineModel` containing preprocessing and regression; and
- writes a reproducibility manifest containing parameters, data coverage, metrics, and timings.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import sys
import time
import uuid

from delta.tables import DeltaTable
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    RobustScaler,
    SQLTransformer,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import LinearRegression
from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAINING_PATH = PROJECT_ROOT / "data" / "processed" / "ml" / "fare_training_dataset"
MODEL_PATH = PROJECT_ROOT / "data" / "models" / "fare_prediction_pipeline"
METRICS_PATH = PROJECT_ROOT / "artifacts" / "week4_fare_metrics.json"

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session

# Keep both heap and native-memory pressure below the Windows page-file limit.
# Evaluation is deliberately streaming and does not cache transformed rows.
spark = build_spark_session(
    "Week4-Task3-Reproducible-Fare-Pipeline",
    driver_memory="4g",
    shuffle_partitions=8,
    master="local[2]",
)
spark.conf.set("spark.sql.files.maxPartitionBytes", 64 * 1024 * 1024)

assert spark.sparkContext.master == "local[2]", (
    "An older SparkContext is still active. Restart the notebook kernel, then "
    "run Task 3 from the first cell so the local[2] resource limit is applied."
)

print("Training input:", TRAINING_PATH)
print("Complete model output:", MODEL_PATH)
print("Metrics output:", METRICS_PATH)
print("Spark master:", spark.sparkContext.master)

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Training input: D:\Master\data intensive\data-intensive-computing\data\processed\ml\fare_training_dataset
Complete model output: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Metrics output: D:\Master\data intensive\data-intensive-computing\artifacts\week4_fare_metrics.json
Spark master: local[2]


## 1. Load and validate the Task 1 dataset

The model predicts the non-negative base fare in US dollars. A regularized linear model is appropriate for this assignment because the objective is a transparent and reproducible workflow rather than maximum predictive accuracy. It also matches Task 2's robust scaling and one-hot encoding design.

The chronological split is loaded from the partitioned Delta table. No random split is performed in this notebook.

In [2]:
EXPECTED_SPLITS = {"train", "validation", "test"}

fare_data = spark.read.format("delta").load(str(TRAINING_PATH))
required_columns = {
    "trip_id", "pickup_datetime", "split", "label",
    "pickup_zone", "dropoff_zone", "pickup_borough", "dropoff_borough",
    "trip_distance", "passenger_count", "vendor_id", "rate_code_id",
    "store_and_fwd_flag", "pickup_month_of_year", "is_weekend",
    "pickup_hour_sin", "pickup_hour_cos", "pickup_dow_sin", "pickup_dow_cos",
    "weather_temperature_c", "weather_relative_humidity_pct",
    "weather_precipitation_mm", "weather_wind_speed_kmh",
    "weather_pressure_hpa", "weather_condition_code",
    "air_quality_pm25_mean", "air_quality_missing",
}
missing_columns = sorted(required_columns - set(fare_data.columns))
assert not missing_columns, f"Task 1 dataset is missing columns: {missing_columns}"

# Task 1 already verifies trip_id uniqueness. Repeating countDistinct over every
# trip here creates an unnecessary high-memory shuffle, so Task 3 checks only
# the contract needed for model training.
split_profile_rows = (
    fare_data.groupBy("split")
    .agg(
        F.count("*").alias("row_count"),
        F.min("pickup_datetime").alias("first_pickup"),
        F.max("pickup_datetime").alias("last_pickup"),
        F.avg("label").alias("mean_label"),
        F.sum(
            (
                F.col("label").isNull()
                | F.isnan("label")
                | (F.col("label") < 0)
            ).cast("long")
        ).alias("invalid_labels"),
    )
    .orderBy("split")
    .collect()
)

assert {row["split"] for row in split_profile_rows} == EXPECTED_SPLITS
assert all(row["row_count"] > 0 for row in split_profile_rows)
assert all(row["invalid_labels"] == 0 for row in split_profile_rows)

split_profile = {
    row["split"]: {
        "row_count": row["row_count"],
        "first_pickup": row["first_pickup"].isoformat(),
        "last_pickup": row["last_pickup"].isoformat(),
        "mean_label": row["mean_label"],
    }
    for row in split_profile_rows
}

# The table is partitioned by split, so these filters are partition-pruned.
# Do not cache the full 9M-row source: each phase reads only the split it needs.
train_raw = fare_data.filter(F.col("split") == "train")
validation_raw = fare_data.filter(F.col("split") == "validation")
test_raw = fare_data.filter(F.col("split") == "test")

print(json.dumps(split_profile, indent=2))

{
  "test": {
    "row_count": 1783894,
    "first_pickup": "2024-03-16T05:00:00",
    "last_pickup": "2024-04-01T05:59:59",
    "mean_label": 19.412305108935886
  },
  "train": {
    "row_count": 5870362,
    "first_pickup": "2024-01-01T06:00:00",
    "last_pickup": "2024-03-01T05:59:58",
    "mean_label": 18.54631074880934
  },
  "validation": {
    "row_count": 1726201,
    "first_pickup": "2024-03-01T06:00:00",
    "last_pickup": "2024-03-16T04:59:59",
    "mean_label": 19.015878956158584
  }
}


## 2. Recreate the reusable Task 2 feature stages

Every stage below matches the Task 2 notebook. Median imputation, category ordering, one-hot dimensions, and robust-scaling statistics are learned only when the complete pipeline is fitted on `train_raw`. `handleInvalid="keep"` allows future data to contain previously unseen categories.

In [3]:
imputer_input_cols = [
    "passenger_count",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "air_quality_pm25_mean",
]
imputer_output_cols = [f"{column}_imputed" for column in imputer_input_cols]

continuous_numeric_cols = [
    "trip_distance",
    "passenger_count_imputed",
    "pickup_month_of_year",
    "weather_temperature_c",
    "weather_relative_humidity_pct_imputed",
    "weather_precipitation_mm_imputed",
    "weather_wind_speed_kmh",
    "weather_pressure_hpa",
    "air_quality_pm25_mean_imputed",
]
binary_cols = ["is_weekend", "air_quality_missing"]
cyclical_cols = [
    "pickup_hour_sin", "pickup_hour_cos", "pickup_dow_sin", "pickup_dow_cos"
]
categorical_input_cols = [
    "pickup_zone_cat", "dropoff_zone_cat", "pickup_borough_cat",
    "dropoff_borough_cat", "vendor_id_cat", "rate_code_id_cat",
    "store_and_fwd_flag_cat", "weather_condition_code_cat",
]
categorical_index_cols = [f"{column}_index" for column in categorical_input_cols]
categorical_vector_cols = [f"{column}_vector" for column in categorical_input_cols]

prepare_categories = SQLTransformer(statement="""
    SELECT
        *,
        COALESCE(pickup_zone, 'Unknown') AS pickup_zone_cat,
        COALESCE(dropoff_zone, 'Unknown') AS dropoff_zone_cat,
        COALESCE(pickup_borough, 'Unknown') AS pickup_borough_cat,
        COALESCE(dropoff_borough, 'Unknown') AS dropoff_borough_cat,
        COALESCE(CAST(vendor_id AS STRING), 'Unknown') AS vendor_id_cat,
        COALESCE(CAST(rate_code_id AS STRING), 'Unknown') AS rate_code_id_cat,
        COALESCE(store_and_fwd_flag, 'Unknown') AS store_and_fwd_flag_cat,
        COALESCE(CAST(weather_condition_code AS STRING), 'Unknown') AS weather_condition_code_cat
    FROM __THIS__
""")

numeric_imputer = Imputer(
    strategy="median",
    inputCols=imputer_input_cols,
    outputCols=imputer_output_cols,
)
category_indexer = StringIndexer(
    inputCols=categorical_input_cols,
    outputCols=categorical_index_cols,
    handleInvalid="keep",
    stringOrderType="alphabetAsc",
)
category_encoder = OneHotEncoder(
    inputCols=categorical_index_cols,
    outputCols=categorical_vector_cols,
    handleInvalid="keep",
    dropLast=True,
)
continuous_assembler = VectorAssembler(
    inputCols=continuous_numeric_cols,
    outputCol="continuous_features_raw",
    handleInvalid="keep",
)
continuous_scaler = RobustScaler(
    inputCol="continuous_features_raw",
    outputCol="continuous_features_scaled",
    lower=0.25,
    upper=0.75,
    withCentering=False,
    withScaling=True,
)
final_assembler = VectorAssembler(
    inputCols=[
        "continuous_features_scaled", *binary_cols, *cyclical_cols,
        *categorical_vector_cols,
    ],
    outputCol="features",
    handleInvalid="keep",
)

feature_stages = [
    prepare_categories,
    numeric_imputer,
    category_indexer,
    category_encoder,
    continuous_assembler,
    continuous_scaler,
    final_assembler,
]

for position, stage in enumerate(feature_stages, start=1):
    print(f"{position}. {stage.__class__.__name__}")

1. SQLTransformer
2. Imputer
3. StringIndexer
4. OneHotEncoder
5. VectorAssembler
6. RobustScaler
7. VectorAssembler


## 3. Train one complete Spark ML pipeline

The saved model contains both feature engineering and regression stages. This is important for reproducibility: future input rows cannot accidentally use different category mappings, medians, scaling statistics, or feature-vector ordering.

The parameters are deliberately fixed and recorded. Retraining means rebuilding Task 1 with new integrated data and rerunning this notebook with the same cells.

In [4]:
REG_PARAM = 0.1
ELASTIC_NET_PARAM = 0.0
MAX_ITER = 50

regressor = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    regParam=REG_PARAM,
    elasticNetParam=ELASTIC_NET_PARAM,
    maxIter=MAX_ITER,
    solver="l-bfgs",
    standardization=False,  # Continuous values are already robustly scaled.
    loss="squaredError",
)

complete_pipeline = Pipeline(stages=[*feature_stages, regressor])
run_id = str(uuid.uuid4())
run_started = time.perf_counter()
fit_started = time.perf_counter()
complete_pipeline_model = complete_pipeline.fit(train_raw)
fit_seconds = time.perf_counter() - fit_started

# Persist the successfully fitted model before evaluation starts. If a later
# reporting step is interrupted, the expensive fit does not need to be lost.
save_started = time.perf_counter()
complete_pipeline_model.write().overwrite().save(str(MODEL_PATH))
save_seconds = time.perf_counter() - save_started

print("Run ID:", run_id)
print("Fit seconds:", round(fit_seconds, 4))
print("Save seconds:", round(save_seconds, 4))
print("Model saved to:", MODEL_PATH)
print("Fitted stages:")
for position, stage in enumerate(complete_pipeline_model.stages, start=1):
    print(f"{position}. {stage.__class__.__name__}")

Run ID: e19359b6-ab35-441b-b305-ff66ff40037d
Fit seconds: 310.6337
Save seconds: 6.4484
Model saved to: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Fitted stages:
1. SQLTransformer
2. ImputerModel
3. StringIndexerModel
4. OneHotEncoderModel
5. VectorAssembler
6. RobustScalerModel
7. VectorAssembler
8. LinearRegressionModel


## 4. Evaluate Validation and Test

MAE is directly interpretable in dollars, RMSE gives additional weight to large fare errors, and R² measures explained variation. The Train-mean baseline shows whether the pipeline improves on a trivial reproducible predictor. Test is used only for final reporting, not parameter selection.

In [5]:
def evaluate_split(model, raw_split, baseline_prediction: float):
    """Compute model and baseline metrics in one streaming Spark aggregation."""
    scored = model.transform(raw_split).select("label", "prediction")
    row = scored.agg(
        F.count("*").alias("n"),
        F.sum(F.abs(F.col("label") - F.col("prediction"))).alias("model_abs"),
        F.sum(F.pow(F.col("label") - F.col("prediction"), 2)).alias("model_sq"),
        F.sum(F.abs(F.col("label") - F.lit(baseline_prediction))).alias("baseline_abs"),
        F.sum(F.pow(F.col("label") - F.lit(baseline_prediction), 2)).alias("baseline_sq"),
        F.sum("label").alias("label_sum"),
        F.sum(F.pow(F.col("label"), 2)).alias("label_sq_sum"),
    ).first()

    n = row["n"]
    total_variance = row["label_sq_sum"] - (row["label_sum"] ** 2 / n)

    def metrics_from(abs_error, squared_error):
        return {
            "mae": abs_error / n,
            "rmse": math.sqrt(squared_error / n),
            "r2": 1.0 - squared_error / total_variance,
        }

    return (
        metrics_from(row["model_abs"], row["model_sq"]),
        metrics_from(row["baseline_abs"], row["baseline_sq"]),
    )


evaluation_started = time.perf_counter()
train_mean_label = float(split_profile["train"]["mean_label"])
validation_metrics, baseline_validation_metrics = evaluate_split(
    complete_pipeline_model, validation_raw, train_mean_label
)
test_metrics, baseline_test_metrics = evaluate_split(
    complete_pipeline_model, test_raw, train_mean_label
)
evaluation_seconds = time.perf_counter() - evaluation_started

metrics = {
    "model": {
        "validation": validation_metrics,
        "test": test_metrics,
    },
    "train_mean_baseline": {
        "prediction": train_mean_label,
        "validation": baseline_validation_metrics,
        "test": baseline_test_metrics,
    },
}

print(json.dumps(metrics, indent=2))
(
    complete_pipeline_model.transform(test_raw.limit(10))
    .select("trip_id", "pickup_datetime", "label", "prediction")
    .show(10, truncate=False)
)

{
  "model": {
    "validation": {
      "mae": 6.806176455922843,
      "rmse": 10.54871987925119,
      "r2": 0.6118146717803717
    },
    "test": {
      "mae": 6.996944909147219,
      "rmse": 10.810044551258805,
      "r2": 0.617767191813297
    }
  },
  "train_mean_baseline": {
    "prediction": 18.54631074880934,
    "validation": {
      "mae": 10.898993065847153,
      "rmse": 16.93741645174195,
      "r2": -0.0007691959106213009
    },
    "test": {
      "mae": 11.190630015547177,
      "rmse": 17.506346495024122,
      "r2": -0.002453031474799694
    }
  }
}
+----------------------------------------------------------------+-------------------+-----+------------------+
|trip_id                                                         |pickup_datetime    |label|prediction        |
+----------------------------------------------------------------+-------------------+-----+------------------+
|000049c7a032845bb215be148d5f7b2fc3f229d84ca460db728a20edb168ea6b|2024-03-19 11:38:43|

In [6]:
for split in ("validation", "test"):
    model_result = metrics["model"][split]
    baseline_result = metrics["train_mean_baseline"][split]
    mae_reduction = 100.0 * (
        baseline_result["mae"] - model_result["mae"]
    ) / baseline_result["mae"]
    rmse_reduction = 100.0 * (
        baseline_result["rmse"] - model_result["rmse"]
    ) / baseline_result["rmse"]
    print(
        f"{split}: MAE reduced by {mae_reduction:.2f}%, "
        f"RMSE reduced by {rmse_reduction:.2f}%, "
        f"R²={model_result['r2']:.3f}"
    )

mae_gap_pct = 100.0 * (
    metrics["model"]["test"]["mae"]
    - metrics["model"]["validation"]["mae"]
) / metrics["model"]["validation"]["mae"]
print(f"Validation-to-test MAE increase: {mae_gap_pct:.2f}%")

validation: MAE reduced by 37.55%, RMSE reduced by 37.72%, R²=0.612
test: MAE reduced by 37.47%, RMSE reduced by 38.25%, R²=0.618
Validation-to-test MAE increase: 2.80%


## 5. Write the reproducibility manifest

The complete model was saved immediately after fitting so an evaluation failure cannot discard the expensive training result. The model directory is overwritten intentionally during retraining. The manifest records the exact Delta version, date coverage, feature groups, model parameters, Spark configuration, metrics, and phase timings needed to explain or reproduce the run.

In [7]:
source_version = int(
    DeltaTable.forPath(spark, str(TRAINING_PATH))
    .history(1)
    .select("version")
    .first()["version"]
)
spark_conf = spark.sparkContext.getConf()

run_manifest = {
    "run_id": run_id,
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "prediction_task": "fare_amount_regression",
    "target": "fare_amount_usd",
    "algorithm": "Spark LinearRegression",
    "input": {
        "path": str(TRAINING_PATH),
        "delta_version": source_version,
        "splits": split_profile,
    },
    "features": {
        "continuous": continuous_numeric_cols,
        "binary": binary_cols,
        "cyclical": cyclical_cols,
        "categorical": categorical_input_cols,
    },
    "hyperparameters": {
        "reg_param": REG_PARAM,
        "elastic_net_param": ELASTIC_NET_PARAM,
        "max_iter": MAX_ITER,
        "solver": "l-bfgs",
        "standardization": False,
    },
    "metrics": metrics,
    "timings_seconds": {
        "fit": round(fit_seconds, 4),
        "evaluate": round(evaluation_seconds, 4),
        "save": round(save_seconds, 4),
        "end_to_end": round(time.perf_counter() - run_started, 4),
    },
    "model_path": str(MODEL_PATH),
    "pipeline_stages": [
        stage.__class__.__name__ for stage in complete_pipeline_model.stages
    ],
    "spark": {
        "version": spark.version,
        "master": spark_conf.get("spark.master", ""),
        "driver_memory": spark_conf.get("spark.driver.memory", ""),
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "session_time_zone": spark.conf.get("spark.sql.session.timeZone"),
    },
}

METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True, default=str) + "\n",
    encoding="utf-8",
)

print("Complete PipelineModel saved to:", MODEL_PATH)
print("Run manifest saved to:", METRICS_PATH)

Complete PipelineModel saved to: D:\Master\data intensive\data-intensive-computing\data\models\fare_prediction_pipeline
Run manifest saved to: D:\Master\data intensive\data-intensive-computing\artifacts\week4_fare_metrics.json


## 6. Verify loading and document retraining

Loading the saved object proves that preprocessing and regression were persisted together. For retraining, refresh the integrated Delta platform, rerun Task 1 to rebuild chronological splits, and rerun this notebook. No feature mappings or imputation values are manually reused from the previous run.

In [8]:
reloaded_model = PipelineModel.load(str(MODEL_PATH))
reloaded_sample = (
    reloaded_model.transform(test_raw.limit(100))
    .select("trip_id", "label", "prediction")
)
assert reloaded_sample.count() == 100
assert reloaded_sample.filter(
    F.col("prediction").isNull() | F.isnan("prediction")
).count() == 0

print("Reloaded model stages:", len(reloaded_model.stages))
print("Reloaded model produced 100 valid predictions.")

Reloaded model stages: 8
Reloaded model produced 100 valid predictions.


## 7. Reusability and extension discussion

### Which parts are reusable?

- The input validation and chronological split contract can be reused for every retraining run.
- The Task 2 feature stages are independent of the regressor and can be reused with another Spark estimator.
- The evaluation function works with any regression model that produces `label` and `prediction`.
- The saved `PipelineModel` applies exactly the same preprocessing and prediction logic to future rows.
- The JSON manifest provides a consistent experiment record for Task 4.

### How can new features be incorporated?

Add the source column to Task 1, register it in the appropriate numerical, binary, cyclical, or categorical feature group, and rebuild the corresponding assembler. Learned preprocessing must still be fitted on Train only. Features derived from post-trip monetary values must not be added because they leak the fare target.

### How is retraining supported?

Weeks 1–3 validate and refresh the integrated Delta table. Task 1 deterministically regenerates the training table, after which rerunning this notebook refits every learned preprocessing stage and the model before overwriting the saved pipeline. The recorded Delta version distinguishes training runs.

### How could multiple prediction tasks be supported?

Keep the loading, split validation, feature stages, evaluator, persistence, and manifest pattern, while replacing the task-specific target, feature groups, and estimator. Trip-duration and fare prediction use regression evaluators; a classification task would substitute a classifier and classification evaluator. Separate model/output paths should be used per task.

## 8. Task 4 hand-off

Task 3 does not implement the Task 4 comparison. It only preserves reusable evidence in `artifacts/week4_fare_metrics.json`: the source Delta version, split sizes and date coverage, Spark configuration, model parameters, evaluation metrics, and separate fit/evaluation/save timings. The Task 4 workflow can use this manifest when comparing the same model and training schema across raw and platform-built datasets.

In [ ]:
# Task 4 intentionally remains a separate workflow.

The recorded `fit` timing can be reused as the Task 3 platform-based training reference. A fair Task 4 experiment should still rerun both approaches under the same Spark configuration rather than treating this single run as a complete benchmark.